# Семинар: Почему Глубокие Сети Нуждаются в Нормализации (Реальный датасет)

**Цель семинара:** На реальном, сложном датасете убедиться, что глубокие нейронные сети без специальных техник не обучаются. Затем, шаг за шагом, применить **Batch Normalization**, **Weight Normalization** и **Residual Connections**, чтобы заставить сеть работать.

**Датасет:** [Covertype](https://archive.ics.uci.edu/ml/datasets/covertype) — задача предсказания типа лесного покрова на основе картографических данных. Это многоклассовая задача (7 классов) с 54 признаками разного масштаба.

## Часть 0: Подготовка

Сначала импортируем все необходимые библиотеки и настроим окружение.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt

# Зафиксируем seed для воспроизводимости
torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

### Загрузка и обработка датасета Covertype

In [ ]:
# --- Этот код мы предоставляем готовым ---

# Загружаем датасет
print("Fetching covtype dataset...")
X, y = fetch_covtype(return_X_y=True)
print("Dataset fetched.")

# ВАЖНО: CrossEntropyLoss в PyTorch ожидает классы от 0 до N-1.
# В этом датасете классы от 1 до 7. Приводим их к нужному диапазону.
y = y - 1

# Разделяем на обучающую и валидационную выборки
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Масштабируем ВХОДНЫЕ данные. Это стандартная практика.
# Это НЕ то же самое, что Batch Normalization, которая нормализует ВНУТРЕННИЕ активации.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

# Конвертируем в тензоры PyTorch
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long) # Для CrossEntropyLoss нужен Long
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

# Создаем датасеты и загрузчики данных
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

BATCH_SIZE = 1024 # Используем батч побольше для ускорения
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

N_FEATURES = X_train.shape[1]
N_CLASSES = len(np.unique(y))
print(f"Размер обучающей выборки: {len(X_train)}")
print(f"Размер валидационной выборки: {len(X_val)}")
print(f"Количество признаков: {N_FEATURES}")
print(f"Количество классов: {N_CLASSES}")

### Цикл обучения и вспомогательные функции

In [ ]:
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(title, fontsize=16)
    
    ax1.plot(history['train_loss'], label='Train Loss')
    ax1.plot(history['val_loss'], label='Validation Loss')
    ax1.set_title('Loss History')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1._get_lines.get_next_color()
    ax1.legend()
    ax1.grid(True)

    ax2.plot(history['train_acc'], label='Train Accuracy')
    ax2.plot(history['val_acc'], label='Validation Accuracy')
    ax2.set_title('Accuracy History')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True)
    
    plt.show()

# ЗАДАНИЕ: Напишите функцию для расчета точности для многоклассовой классификации
def calculate_accuracy_multiclass(y_pred, y_true):
    # Подсказка: y_pred - это логиты (сырые выходы сети) размера (batch_size, num_classes)
    # 1. Найдите индекс класса с максимальным логитом для каждого примера в батче.
    #    Поможет `torch.argmax`.
    predicted = # ... ваш код здесь
    # 2. Сравните предсказанные индексы с настоящими (y_true).
    # 3. Посчитайте долю правильных ответов.
    correct = # ... ваш код здесь
    return correct / len(y_true)

def train_model(model, title, train_loader, val_loader, epochs, learning_rate):
    print(f"--- Training {title} ---")
    model.to(device)
    
    # ЗАДАНИЕ: Определите функцию потерь и оптимизатор
    # Для многоклассовой классификации с логитами на выходе нужна CrossEntropyLoss
    criterion = # ... ваш код здесь
    optimizer = # ... ваш код здесь (Adam - хороший выбор)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        running_loss, running_acc = 0.0, 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # ЗАДАНИЕ: Напишите стандартный цикл обучения (forward, loss, backward, step)

            # шаг 1: обнуляем градиенты

            # шаг 2: делаем прямой проход

            # шаг 3: считаем loss

            # шаг 4: делаем обратный проход

            # шаг 5: обновляем веса
            
            running_loss += loss.item() * inputs.size(0)
            running_acc += calculate_accuracy_multiclass(outputs, labels) * inputs.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_acc / len(train_loader.dataset)
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)

        model.eval()
        val_loss, val_acc = 0.0, 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                # пишем цикл валидации

                # шаг 1: делаем прямой проход
                outputs = ...

                # шаг 2: считаем loss
                loss = ...

                # шаг 3: считаем accuracy - написано за вас
                val_loss += loss.item() * inputs.size(0)
                val_acc += calculate_accuracy_multiclass(outputs, labels) * inputs.size(0)
        
        val_loss /= len(val_loader.dataset)
        val_acc /= len(val_loader.dataset)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        if (epoch + 1) % 2 == 0 or epoch == 0:
             print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    plot_history(history, title)
    return model

## Часть 1: Проблема глубоких сетей

Построим глубокий MLP без каких-либо нормализаций и посмотрим, как он справится со сложным датасетом.

In [ ]:
class DeepMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(DeepMLP, self).__init__()
        layers = []
        
        # ЗАДАНИЕ: Соберите глубокую сеть из nn.Linear и nn.ReLU
        # 1. Входной слой: преобразует input_size в hidden_size
        # 2. Скрытые слои (num_layers - 2): преобразуют hidden_size в hidden_size
        # 3. Выходной слой: преобразует hidden_size в output_size (без активации!)
        
        # ... ваш код здесь ...

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

### Эксперимент 1: Обучаем глубокую "ванильную" сеть

Попробуем обучить нашу 12-слойную сеть. Если все сделано правильно, обучение должно полностью провалиться.

In [ ]:
HIDDEN_SIZE = 256
NUM_LAYERS = 12 # Глубокая сеть
EPOCHS = 10
LR = 1e-4

# Сначала заполните недостающие части в функциях calculate_accuracy и train_model
vanilla_model = DeepMLP(N_FEATURES, HIDDEN_SIZE, NUM_LAYERS, N_CLASSES)

trained_vanilla_model = train_model(vanilla_model, "Vanilla Deep MLP (12 layers)", train_loader, val_loader, EPOCHS, LR)

**Ожидаемый вывод:** Вы увидите, что обучение не происходит. Ошибка "застрянет" на значении, близком к `ln(7)` ≈ 1.94, а точность будет близка к случайному угадыванию (1/7 ≈ 14%). Это идеальная демонстрация проблемы!

## Часть 2: Batch Normalization

Теперь добавим `nn.BatchNorm1d` после каждого линейного слоя (кроме последнего) и перед активацией.

In [ ]:
class DeepMLP_BN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(DeepMLP_BN, self).__init__()
        layers = []
        
        # ЗАДАНИЕ: Соберите сеть, используя шаблон Linear -> BatchNorm -> ReLU
        # - Для nn.BatchNorm1d нужно указать количество фичей, которое через него проходит (hidden_size).
        # - В линейных слоях перед Batch Norm можно убрать bias (bias=False), т.к. он все равно будет 'обнулен'.
        
        # ... ваш код здесь ...

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

### Эксперимент 2: Обучаем сеть с Batch Norm

Картина должна драматически измениться.

In [ ]:
bn_model = DeepMLP_BN(N_FEATURES, HIDDEN_SIZE, NUM_LAYERS, N_CLASSES)

trained_bn_model = train_model(bn_model, "MLP with Batch Norm", train_loader, val_loader, EPOCHS, LR)

**Ожидаемый вывод:** Обучение пойдет полным ходом! Ошибка будет быстро падать, а точность — расти. Это показывает, насколько мощным инструментом является Batch Normalization.

## Часть 3: Weight Normalization

Рассмотрим альтернативу — `Weight Normalization`, которая не зависит от статистики батча.

In [ ]:
from torch.nn.utils import weight_norm

class DeepMLP_WN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(DeepMLP_WN, self).__init__()
        layers = []
        
        # ЗАДАНИЕ: Соберите сеть, оборачивая каждый nn.Linear в `weight_norm()`.
        # Шаблон: weight_norm(Linear) -> ReLU
        
        # ... ваш код здесь ...

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

### Эксперимент 3: Обучаем сеть с Weight Norm

In [ ]:
wn_model = DeepMLP_WN(N_FEATURES, HIDDEN_SIZE, NUM_LAYERS, N_CLASSES)

trained_wn_model = train_model(wn_model, "MLP with Weight Norm", train_loader, val_loader, EPOCHS, LR)

**Ожидаемый вывод:** Weight Normalization также должна заставить сеть обучаться, хотя динамика может немного отличаться от Batch Norm. Это доказывает, что стабилизация весов — тоже рабочий подход.

## Часть 4: Residual Connections

Теперь построим сеть на основе остаточных блоков. Это золотой стандарт для очень глубоких сетей. Внутри блоков обычно также используется Batch Norm.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, size):
        super(ResidualBlock, self).__init__()
        # ЗАДАНИЕ: Определите внутреннюю структуру блока F(x).
        # Обычно это два слоя с нормализацией и активацией.
        # Шаблон: Linear -> BN -> ReLU -> Linear -> BN
        self.block = nn.Sequential(
            # ... ваш код здесь ...
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # ЗАДАНИЕ: Реализуйте прямой проход H(x) = ReLU(F(x) + x)
        # 1. Пропустите x через self.block, чтобы получить F(x) (или residual)
        # 2. Сложите результат с исходным x.
        # 3. Примените к сумме активацию ReLU.
        return # ... ваш код здесь

class ResNetMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_blocks, output_size):
        super(ResNetMLP, self).__init__()
        # ЗАДАНИЕ: Соберите сеть:
        # 1. Входной слой (Linear) для преобразования input_size -> hidden_size, за которым следует ReLU.
        # 2. Последовательность из num_blocks остаточных блоков (ResidualBlock).
        # 3. Выходной слой (Linear) для преобразования hidden_size -> output_size.
        self.entry = # ... ваш код здесь
        self.blocks = # ... ваш код здесь
        self.output = # ... ваш код здесь
        
    def forward(self, x):
        x = self.entry(x)
        x = self.blocks(x)
        x = self.output(x)
        return x

### Эксперимент 4: Обучаем сеть с Residual Connections

In [ ]:
# Глубина: 1 входной слой + 5 блоков * 2 слоя/блок + 1 выходной = 12 слоев
NUM_BLOCKS = 5 

res_model = ResNetMLP(N_FEATURES, HIDDEN_SIZE, NUM_BLOCKS, N_CLASSES)

trained_res_model = train_model(res_model, "MLP with Residual Connections", train_loader, val_loader, EPOCHS, LR)

**Ожидаемый вывод:** ResNet-подобная архитектура должна показать очень стабильное и эффективное обучение. Это демонстрирует, почему проброс связей так важен для построения действительно глубоких моделей.

## Итоговое заключение

Этот семинар наглядно показал, что простое "наращивание" слоев не работает на сложных задачах. Стабилизирующие техники, такие как Batch/Weight Normalization и архитектурные решения вроде Residual Connections, являются не просто "улучшениями", а **необходимыми компонентами** для успешного обучения глубоких нейронных сетей.